# Qwen3-0.6B G検定 Evaluation — Choice-text scoring

このNotebookはOpenAI評価と同じ `MMLUEvaluator` を使います。
通常は次の **User settings** だけを変更し、GPU runtimeでRun allしてください。

処理順: Settings → Environment setup → Model loading → Dataset / evaluator setup
→ Preflight / manifest → Evaluation → Results


## 1. Settings

### User settings — 通常はこのセルだけを変更

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("/content/AIkenSGTv1_New")
DATA_ROOT = Path("/content/gkentei-reference")
DATA_DIR = DATA_ROOT / "data"

# Evaluation repository
if not (PROJECT_DIR / "mmlu_eval").is_dir():
    !rm -rf /content/AIkenSGTv1_New
    !git clone https://github.com/HayatoHongo/AIkenSGTv1.git /content/AIkenSGTv1_New
    !cd /content/AIkenSGTv1_New && git switch tayama

# G検定データ: eval1=test, eval2=dev
from urllib.request import urlretrieve
import csv

DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "dev").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "test").mkdir(parents=True, exist_ok=True)

DATA_URLS = {
    "test": "https://raw.githubusercontent.com/HayatoHongo/AIkenSGTv1/main/gkentei_eval1.jsonl",
    "dev": "https://raw.githubusercontent.com/HayatoHongo/AIkenSGTv1/main/gkentei_eval2.jsonl",
}

for split, url in DATA_URLS.items():
    jsonl_path = DATA_ROOT / f"gkentei_{split}.jsonl"
    csv_path = DATA_DIR / split / f"gkentei_{split}.csv"
    if not jsonl_path.exists():
        urlretrieve(url, jsonl_path)
    with jsonl_path.open(encoding="utf-8") as source, csv_path.open("w", newline="", encoding="utf-8") as target:
        writer = csv.writer(target)
        for line in source:
            item = json.loads(line)
            correct = chr(ord("A") + int(item["correct_answers"]) - 1)
            writer.writerow([item["question_text"], item["option_1"], item["option_2"], item["option_3"], item["option_4"], correct])

OUTPUT_DIR = Path("/content/drive/MyDrive/qwen3_gkentei_results")

SUBJECT = "gkentei"
NTRAIN = 3
SAMPLE_FRAC = 1.0
SEED = 42
LIMIT = 0
MANIFEST_PATH = None

### Project settings — 通常の問題セット変更では編集不要

In [21]:
MODEL_ID = "Qwen/Qwen3-0.6B"
TOKENIZER = MODEL_ID
MODEL_FORMAT = "hf"

DEVICE = "cuda"
DTYPE = "bfloat16"
BATCH_SIZE = 1
MAX_CONTEXT_LENGTH = 2048

CONTEXT_POLICY = "reduce"
CONTEXT_TOKENIZER = "gpt2"

PERMUTATION_COUNT = 4
PERMUTATION_SEED = 0

SCORING_METHOD = "choice_text"
TEXT_REDUCTION = "mean"
OUTPUT_NAME = f"qwen3_0.6b_gkentei_{NTRAIN}shot_text_{TEXT_REDUCTION}.csv"

## 2. Environment setup

In [22]:
%pip install -q numpy pandas tiktoken transformers safetensors huggingface_hub


In [23]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "mmlu_eval").is_dir(), (
    "PROJECT_DIR must point to the repository containing mmlu_eval/"
)
assert DATA_DIR.is_dir(), "DATA_DIR must contain dev/ and test/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Model loading

In [24]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from mmlu_eval.backends.aikengpt_backend import (
    AIkenGPTBackend,
)

assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a Colab GPU runtime"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=getattr(torch, DTYPE),
).to(DEVICE)
model.config.use_cache = False
print("Loaded:", MODEL_ID)


GPU: NVIDIA L4


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-0.6B


## 4. Dataset / evaluator setup

In [25]:
from mmlu_eval import EvalConfig, MMLUEvaluator

eval_config = EvalConfig(
    ntrain=NTRAIN,
    sample_frac=SAMPLE_FRAC,
    seed=SEED,
    limit=LIMIT,
    subject=SUBJECT,
    permutation_count=PERMUTATION_COUNT,
    permutation_seed=PERMUTATION_SEED,
    context_policy=CONTEXT_POLICY,
    context_tokenizer=CONTEXT_TOKENIZER,
    max_context_length=MAX_CONTEXT_LENGTH,
)
evaluator = MMLUEvaluator(DATA_DIR, eval_config)


## 5. Preflight / manifest

In [26]:
from mmlu_eval.core import atomic_csv

# Existing manifest is authoritative. Otherwise create one once and reuse it below.
manifest = evaluator.manifest(MANIFEST_PATH)
preflight_manifest_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.manifest.csv")
preflight_prompts_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.prompts.jsonl")
atomic_csv(preflight_manifest_path, manifest)
evaluator.export_debug(manifest, preflight_prompts_path)
active_manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH is not None else preflight_manifest_path

first = manifest.iloc[0]
first_case = evaluator.cases(first.subject, int(first.test_index))[0]
print(f"Questions: {len(manifest)} / prompts: {len(manifest) * 4}")
print("Manifest:", active_manifest_path)
print("\nFirst prompt:\n")
print(first_case.prompt)


Questions: 145 / prompts: 580
Manifest: /content/drive/MyDrive/qwen3_gkentei_results/qwen3_0.6b_gkentei_0shot_letter.preflight.manifest.csv

First prompt:

The following are multiple choice questions (with answers) about gkentei.

ディープラーニングの構成要素について、説明と名称の組み合わせとして最も適切なものを1つ選べ。
（あ）画像の局所的なパターンを捉え、位置ごとに同じフィルタを適用するニューラルネットワーク
（い）特徴マップを縮約し、空間方向の解像度を下げる処理
（う）各入力ユニットと出力ユニットを完全に結合し、特徴を統合する層
（え）過去の入力を内部状態として利用し、系列の依存関係を扱うニューラルネットワーク
A. （あ）リカレントニューラルネットワーク、（い）プーリング、（う）全結合層、（え）畳み込みニューラルネットワーク
B. （あ）畳み込みニューラルネットワーク、（い）全結合層、（う）プーリング、（え）リカレントニューラルネットワーク
C. （あ）全結合層、（い）畳み込みニューラルネットワーク、（う）リカレントニューラルネットワーク、（え）プーリング
D. （あ）畳み込みニューラルネットワーク、（い）プーリング、（う）全結合層、（え）リカレントニューラルネットワーク
Answer:


## 6. Evaluation

In [27]:
backend = AIkenGPTBackend(
    model,
    tokenizer,
    model_identifier=MODEL_ID,
    tokenizer_identifier=TOKENIZER,
    scoring_method=SCORING_METHOD,
    text_reduction=TEXT_REDUCTION,
    model_format=MODEL_FORMAT,
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_context_length=MAX_CONTEXT_LENGTH,
    checkpoint_sha256=None,
)

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
results = evaluator.run(
    backend,
    OUTPUT_PATH,
    manifest_path=active_manifest_path,
)


1/145 gkentei[105]
2/145 gkentei[144]
3/145 gkentei[69]
4/145 gkentei[114]
5/145 gkentei[37]
6/145 gkentei[26]
7/145 gkentei[54]
8/145 gkentei[57]
9/145 gkentei[107]
10/145 gkentei[120]
11/145 gkentei[45]
12/145 gkentei[9]
13/145 gkentei[44]
14/145 gkentei[14]
15/145 gkentei[36]
16/145 gkentei[112]
17/145 gkentei[119]
18/145 gkentei[0]
19/145 gkentei[98]
20/145 gkentei[104]
21/145 gkentei[138]
22/145 gkentei[7]
23/145 gkentei[59]
24/145 gkentei[65]
25/145 gkentei[25]
26/145 gkentei[123]
27/145 gkentei[68]
28/145 gkentei[76]
29/145 gkentei[75]
30/145 gkentei[41]
31/145 gkentei[72]
32/145 gkentei[89]
33/145 gkentei[136]
34/145 gkentei[64]
35/145 gkentei[23]
36/145 gkentei[11]
37/145 gkentei[106]
38/145 gkentei[111]
39/145 gkentei[48]
40/145 gkentei[56]
41/145 gkentei[74]
42/145 gkentei[108]
43/145 gkentei[133]
44/145 gkentei[3]
45/145 gkentei[130]
46/145 gkentei[29]
47/145 gkentei[142]
48/145 gkentei[28]
49/145 gkentei[71]
50/145 gkentei[70]
51/145 gkentei[15]
52/145 gkentei[140]
53/145 

## 7. Results

In [28]:
from mmlu_eval.core import summarize

print(summarize(results))
print("Result CSV:", OUTPUT_PATH)
print("Manifest:", OUTPUT_PATH.with_suffix(".manifest.csv"))
print("Prompts:", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
display(results.head())

# OpenAI runとの入力一致を確認する場合:
# from mmlu_eval.compare import compare
# compare("/path/to/openai.prompts.jsonl", OUTPUT_PATH.with_suffix(".prompts.jsonl"))


{'n': 145, 'baseline_accuracy': 0.5586206896551724, 'baseline_recall_A': 0.5151515151515151, 'baseline_recall_B': 0.6097560975609756, 'baseline_recall_C': 0.6304347826086957, 'baseline_recall_D': 0.4, 'baseline_rstd': 0.0911816688421359, 'debiased_accuracy': 0.593103448275862, 'debiased_recall_A': 0.5151515151515151, 'debiased_recall_B': 0.6585365853658537, 'debiased_recall_C': 0.6521739130434783, 'debiased_recall_D': 0.48, 'debiased_rstd': 0.07989434317731973, 'difference': 0.03448275862068961, 'position_A_prob': 0.23165352376834775, 'position_B_prob': 0.25061142629056893, 'position_C_prob': 0.2616403279879715, 'position_D_prob': 0.2560947219531118, 'wrong_to_correct': 12, 'correct_to_wrong': 7}
Result CSV: /content/drive/MyDrive/qwen3_gkentei_results/qwen3_0.6b_gkentei_0shot_letter.csv
Manifest: /content/drive/MyDrive/qwen3_gkentei_results/qwen3_0.6b_gkentei_0shot_letter.manifest.csv
Prompts: /content/drive/MyDrive/qwen3_gkentei_results/qwen3_0.6b_gkentei_0shot_letter.prompts.jsonl


,subject,test_index,question,A,B,C,D,label,effective_ntrain,model_identifier,...,perm3_missing_count,perm3_error_bound,perm3_fifth_logprob,input_tokens,output_tokens,mean_error_bound,max_error_bound,has_all_missing,sample_order,run_id
0,gkentei,105,ディープラーニングの構成要素について、説明と名称の組み合わせとして最も適切なものを1つ選べ。...,（あ）リカレントニューラルネットワーク、（い）プーリング、（う）全結合層、（え）畳み込みニュ...,（あ）畳み込みニューラルネットワーク、（い）全結合層、（う）プーリング、（え）リカレントニュ...,（あ）全結合層、（い）畳み込みニューラルネットワーク、（う）リカレントニューラルネットワーク...,（あ）畳み込みニューラルネットワーク、（い）プーリング、（う）全結合層、（え）リカレントニュ...,D,0,Qwen/Qwen3-0.6B,...,None,None,None,1380,0,None,None,False,0,ff86aa205fb7ebdf4e56f30c15fc440d676485a6fb7bbf...
1,gkentei,144,動画配信サービスで、ある利用者の視聴履歴から好みのジャンルや出演者などを把握し、他の利用者の...,コンテンツベースフィルタリングを用いる,デモグラフィックフィルタリングを用いる,協調フィルタリングを用いる,人気度ベース推薦を用いる,A,0,Qwen/Qwen3-0.6B,...,None,None,None,556,0,None,None,False,1,ff86aa205fb7ebdf4e56f30c15fc440d676485a6fb7bbf...
2,gkentei,69,画像分類モデルの畳み込み層の出力が、高さ14、幅14、チャネル数128の特徴マップである。各...,Flattenを用いる,2×2 Average Poolingを用いる,1×1 Convolutionを用いる,Global Average Pooling（GAP）を用いる,D,0,Qwen/Qwen3-0.6B,...,None,None,None,624,0,None,None,False,2,ff86aa205fb7ebdf4e56f30c15fc440d676485a6fb7bbf...
3,gkentei,114,自然言語処理における文書の数値化手法に関する説明と名称の組み合わせとして、最も適切なものを1...,（あ）TF-IDF、（い）単語埋め込み,（あ）単語埋め込み、（い）TF-IDF,（あ）BoW（Bag-of-Words）、（い）ワンホット表現,（あ）BoW（Bag-of-Words）、（い）単語埋め込み,D,0,Qwen/Qwen3-0.6B,...,None,None,None,808,0,None,None,False,3,ff86aa205fb7ebdf4e56f30c15fc440d676485a6fb7bbf...
4,gkentei,37,人工知能をめぐる動向に関する説明のうち、東ロボくんについての説明を1つ選べ。,大学入試問題への回答を目指して開発された人工知能プロジェクト,大学入試問題を自動的に作成するために開発された人工知能システム,大学の講義内容に応じて学生を指導するために開発された人工知能システム,大学入試の出願者数を予測するために開発された人工知能システム,A,0,Qwen/Qwen3-0.6B,...,None,None,None,580,0,None,None,False,4,ff86aa205fb7ebdf4e56f30c15fc440d676485a6fb7bbf...
